# WHO Disease Outbreak News Web Scraping

## CodeAlpha Data Analytics Internship

**Author:** David Edeh

---

### Project Description

This project demonstrates how to scrape disease outbreak reports from the World Health Organization (WHO) Disease Outbreak News website using Selenium and BeautifulSoup.

The website loads its content dynamically using JavaScript. Therefore, Selenium is used to render the page before BeautifulSoup extracts the outbreak information.

The extracted data is cleaned, validated and exported as a CSV dataset for further exploratory data analysis and visualization.

## Project Objectives

The objectives of this project are to:

- Learn web scraping using Selenium and BeautifulSoup.
- Understand how to inspect HTML structure.
- Handle dynamically rendered web pages.
- Extract outbreak report information.
- Perform automatic pagination.
- Build a structured dataset.
- Export the dataset for analysis.

## Required Libraries

This notebook uses:

- Selenium
- BeautifulSoup
- pandas
- tqdm
- webdriver-manager

In [2]:
import pandas as pd

import logging

import time

from datetime import datetime

from bs4 import BeautifulSoup

from tqdm import tqdm

from selenium import webdriver

from selenium.webdriver.common.by import By

from selenium.webdriver.chrome.service import Service

from selenium.webdriver.support.ui import WebDriverWait

from selenium.webdriver.support import expected_conditions as EC

from webdriver_manager.chrome import ChromeDriverManager

## Why Selenium?

Initially, BeautifulSoup alone was considered for this project.

However, inspecting the HTML source of the WHO website showed that outbreak reports are loaded dynamically using JavaScript.

Because requests and BeautifulSoup cannot execute JavaScript, Selenium was used to render the page before extracting the HTML.

This workflow allows BeautifulSoup to parse the fully rendered HTML.

In [3]:
options = webdriver.ChromeOptions()

options.add_argument("--start-maximized")

driver = webdriver.Chrome(
    service=Service(
        ChromeDriverManager().install()
    ),
    options=options
)

## Open WHO Website
Via the driver.get() command, Selenium is used to launch the browser to the BASE_URL and allows the entire page to load before returning control to the script. Wiating is necessary to ensure complete loading before returning control to the script

In [7]:
WAIT_TIME = 20
PAGE_LOAD_DELAY = 3

def wait_for_page(driver):

    WebDriverWait(
        driver,
        WAIT_TIME
    ).until(
        EC.presence_of_element_located(
            (By.TAG_NAME, "body")
        )
    )

    time.sleep(PAGE_LOAD_DELAY)

BASE_URL = "https://www.who.int/emergencies/disease-outbreak-news"

driver.get(BASE_URL)

wait_for_page(driver)

## Capture Rendered HTML
Selenium executes JavaScript and captures the rendered HTML of the WHO Disease Outbreak News website.

In [8]:
rendered_html = driver.page_source

## Parse HTML
BeautifulSoup is then used to parse the rendered HTML

In [9]:
soup = BeautifulSoup(
    rendered_html,
    "html.parser"
)

### HTML Inspection

After rendering the page, the HTML was inspected using the browser developer tools.

The rendered outbreak reports were found inside HTML elements matching the CSS selector:

```css
a.sf-list-vertical__item

In [10]:
reports = soup.select(
    "a.sf-list-vertical__item"
)

len(reports)

28

In [12]:
report = reports[0]

def extract_report(report):
    
    title_container = report.select_one("h4.sf-list-vertical__title")

    title_element = title_container.select_one("span.trimmed")

    report_title = (
        title_element.get_text(strip=True)
        if title_element
        else "Unknown Title"
    )

    spans = title_container.find_all("span")

    publication_date = ""
    
    if len(spans) >= 2:
        publication_date = (
            spans[1]
            .get_text(strip=True)
            .replace("|", "")
            .strip()
        )

    report_url = report.get("href")

    from urllib.parse import urljoin

    report_url = urljoin(BASE_URL, report.get("href"))

    record = {
        "report_title": report_title,
        "publication_date": publication_date,
        "report_url": report_url
    }
    return record

In [13]:
records = []

for report in reports:

    record = extract_report(report)
    records.append(record)

## Pagination
Pagination was present on the site, and based on my understanding of HTML, I wrote the code that helps Selenium interact with the next button. The scraper was configured to stop once it reaches reports older than 2020

In [14]:
df = pd.DataFrame(records)
df.head()
df.info()
df.describe(include="all")

<class 'pandas.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   report_title      28 non-null     str  
 1   publication_date  28 non-null     str  
 2   report_url        28 non-null     str  
dtypes: str(3)
memory usage: 804.0 bytes


,report_title,publication_date,report_url
count,28,28,28
unique,21,26,28
top,"Ebola disease caused by Bundibugyo virus, Demo...",5 December 2025,https://www.who.int/emergencies/disease-outbre...
freq,5,2,1


In [15]:
df.to_csv(
    "who_outbreak_reports.csv",
    index=False
)

## Project Summary

Successfully extracted outbreak reports.

The dataset includes:

- Report Title
- Publication Date
- Disease
- Country
- Year
- Month
- Report URL

The dataset is ready for exploratory data analysis.

## Key Learnings

During this project I learned:

- The difference between static and dynamic web pages.
- Why Selenium is required for JavaScript-rendered websites.
- How BeautifulSoup parses HTML.
- How to inspect HTML using browser developer tools.
- How pagination works.
- How to build reusable scraping functions.
- The importance of exception handling and logging.
- How to export structured datasets for further analysis.

## Next Steps

The exported dataset will be used for:

- Exploratory Data Analysis (EDA)
- Data Cleaning
- Data Visualization
- Trend Analysis

These analyses form the subsequent tasks of the CodeAlpha Data Analytics Internship.